In [2]:
!pip --version

pip 23.3 from /opt/conda/lib/python3.11/site-packages/pip (python 3.11)


In [3]:
!pip install pyspark==3.5.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 14.1 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.1 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.3-py2.py3-none-any.whl size=317840630 sha256=3377b9a61d942f9c5e374ce3c160acfdedc5a5f7adc4b9ef2a7dab7969b1dc10
  Stored in directory: /tmp/pip-ephem-wheel-cache-v7o7ea4b/wheels/97/f5/c0/947e2c0942b361ffe58651f36bd7f13772675b3863fd63d1b1
Successfully built pyspark


# 1. Spark RDD

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession \
    .builder \
    .appName("test_app") \
    .getOrCreate()

In [6]:
spark

In [12]:
# создаём рабочую директорию и кладём входной файл
! hdfs dfs -mkdir -p /hadoop/hw2/task1/input
! hdfs dfs -put bible.txt /hadoop/hw2/task1/input

put: `/hadoop/hw2/task1/input/bible.txt': File exists


In [14]:
# создаём rdd по файлу из HDFS
rdd = spark.sparkContext.textFile("/hadoop/hw2/task1/input/bible.txt")

word_counts = (rdd
    # (transform) разбиваем входной текст на слова по пробелам и переносам строк
    .flatMap(lambda line: line.split())
    # (transform) оставляем только те слова, длина которых больше 4
    .filter(lambda word: len(word) > 4)
    # (transform) каждое слово трансформируем в кортеж из этой слова и единицы
    .map(lambda word: (word, 1))
    # (transform) для всех элементов с одинаковыми словами суммируются их значения
    .reduceByKey(lambda a, b: a + b)
    # (action) берём 10 слов, которые встречаются чаще всего
    .takeOrdered(10, key=lambda kv: -kv[1])
)

for word, count in word_counts:
    print(f"{word}\t{count}")

shall	9658
which	4119
their	3801
there	1799
before	1701
against	1587
shalt	1586
children	1559
said,	1556
them,	1549


# 2. Spark DataFrame

In [15]:
import pyspark.sql.functions as sf

In [21]:
# создаём DataFrame с одним столбцом "value" (каждое значение в таблице - отдельная строка файла) 
df = spark.read.text("/hadoop/hw2/task1/input/bible.txt")
df.show(5)

+--------------------+
|               value|
+--------------------+
|In the beginning ...|
|And God said, Let...|
|And God saw the l...|
|And God called th...|
|And God said, Let...|
+--------------------+
only showing top 5 rows



In [26]:
words_df = (df
    # split(col("value"), r"\s+") - разбиваем строку колонки value на массив слов
    # explode(...) - разбивает элементы массива по отдельным строкам (аналог flatMap из RDD)
    # alias("word") - переименовываем столбец в "word"
    # все они (transform)
    .select(sf.explode(sf.split(sf.col("value"), r"\s+")).alias("word"))
    # (transform) оставляем строки с длиной символов больше 4
    .filter(sf.length(sf.col("word")) > 4)
    # (transform) группируем строки по значению столбца "word"
    .groupBy("word")
    # (transform) для каждой группы считаем кол-во строк и это число добавляется в новый столбец "count"
    .count()
    # (transform) сортируем по столбцу "count" и по убыванию
    .orderBy(sf.desc("count"))
)

words_df.show(10)

+--------+-----+
|    word|count|
+--------+-----+
|   shall| 9658|
|   which| 4119|
|   their| 3801|
|   there| 1799|
|  before| 1701|
| against| 1587|
|   shalt| 1586|
|children| 1559|
|   said,| 1556|
|   them,| 1549|
+--------+-----+
only showing top 10 rows



# Проверка результатов с MapReduce

Сначала отсортируем результат от MapReduce по убыванию кол-ва найденных слов.

In [27]:
! sort -t $'\t' -k2 -nr hadoop_result.txt | head -10

shall	9658
which	4119
their	3801
there	1799
before	1701
against	1587
shalt	1586
children	1559
said,	1556
them,	1549
sort: write failed: 'standard output': Broken pipe
sort: write error


### В итоге видим, что все 3 способа дали одинаковые результаты ("shall"=9658 и "them,"=1549). Значит всё было выполнено верно.